In [28]:
import pandas as pd
import numpy as np

# Dataset 1: Raw Sales Transactions (Sales_System)
sales_data = {
    "order_id": ["101", "102", "103", "104", "105", "105", "106", "107"],
    "customer_id": ["C10", "C11", "C12", "C10", "C13", "C13", "C14", "C15"],
    "product_sku": ["PROD_A", "PROD_B", "PROD_A", "PROD_C", "PROD_B", "PROD_B", "PROD_A", "PROD_D"],
    "quantity": [2, "1", 3, None, 1, 1, -1, 5],  # Contains bad data (string, NaN, negative)
    "unit_price": ["$120.00", "$45.50", "120.00", "$300.00", "$45.50", "$45.50", "$120.00", "$15.00"],
    "transaction_date": ["2026-01-05", "2026/01/06", "Jan 07, 2026", "2026-01-08", "2026-01-08", "2026-01-08", "2026-01-09", "2026-01-10"]
}

# Dataset 2: Customer Profiles (CRM_System)
customer_data = {
    "customer_id": ["C10", "C11", "C12", "C13", "C14", "C16"],
    "country": ["USA", "USA", "Canada", " UK ", "USA", "Germany"],
    "signup_date": ["2025-05-12", "2025-08-19", "2025-11-01", "2026-01-01", "2025-02-14", "2026-01-02"]
}

df_sales = pd.DataFrame(sales_data)
df_customers = pd.DataFrame(customer_data)

In [44]:
sum_df_sales = pd.DataFrame({
    'missing': df_sales.isnull().sum(),
    'unique': df_sales.nunique(),
    'typr': df_sales.dtypes
    
})

sum_df_cust = pd.DataFrame({
    'missing': df_customers.isnull().sum(),
    'unique': df_customers.nunique(),
    'type': df_customers.dtypes
})

sum_df_sales


,missing,unique,typr
order_id,0,6,str
customer_id,0,6,str
product_sku,0,3,str
quantity,0,5,int64
unit_price,0,4,float64
transaction_date,1,5,datetime64[us]


In [34]:
df_sales = df_sales.dropna()
df_sales['quantity'] = df_sales['quantity'].astype(int)

In [35]:
df_sales['transaction_date'] = df_sales['transaction_date'].str.replace('/', '-')
df_sales['unit_price'] = df_sales['unit_price'].str.replace('$', '')
df_sales['quantity'] = np.where(df_sales['quantity'] < 0, 0, df_sales['quantity'])
df_sales

,order_id,customer_id,product_sku,quantity,unit_price,transaction_date
0,101,C10,PROD_A,2,120.00,2026-01-05
1,102,C11,PROD_B,1,45.50,2026-01-06
2,103,C12,PROD_A,3,120.00,"Jan 07, 2026"
4,105,C13,PROD_B,1,45.50,2026-01-08
5,105,C13,PROD_B,1,45.50,2026-01-08
6,106,C14,PROD_A,0,120.00,2026-01-09
7,107,C15,PROD_D,5,15.00,2026-01-10


In [46]:
df_sales['transaction_date'] = pd.to_datetime(df_sales['transaction_date'], errors='coerce')
df_sales['unit_price'] = np.where(df_sales['quantity'] == 0, 0, df_sales['unit_price'])
df_sales['unit_price'] = df_sales['unit_price'].astype(float)
df_sales = df_sales[df_sales['quantity'] != 0]


df_sales


,order_id,customer_id,product_sku,quantity,unit_price,transaction_date
0,101,C10,PROD_A,2,120.0,2026-01-05
1,102,C11,PROD_B,1,45.5,2026-01-06
2,103,C12,PROD_A,3,120.0,NaT
4,105,C13,PROD_B,1,45.5,2026-01-08
5,105,C13,PROD_B,1,45.5,2026-01-08
7,107,C15,PROD_D,5,15.0,2026-01-10


In [49]:
df_customers['country'] = df_customers['country'].str.strip()
df_sales = df_sales.drop_duplicates()
df_sales

,order_id,customer_id,product_sku,quantity,unit_price,transaction_date
0,101,C10,PROD_A,2,120.0,2026-01-05
1,102,C11,PROD_B,1,45.5,2026-01-06
2,103,C12,PROD_A,3,120.0,NaT
4,105,C13,PROD_B,1,45.5,2026-01-08
7,107,C15,PROD_D,5,15.0,2026-01-10


In [99]:
df_sales['total'] = df_sales['quantity'] * df_sales['unit_price']
df_sales

,order_id,customer_id,product_sku,quantity,unit_price,transaction_date,total,month,day
0,101,C10,PROD_A,2,120.0,2026-01-05,240.0,January,Monday
1,102,C11,PROD_B,1,45.5,2026-01-06,45.5,January,Tuesday
2,103,C12,PROD_A,3,120.0,NaT,360.0,NaN,NaN
4,105,C13,PROD_B,1,45.5,2026-01-08,45.5,January,Thursday
7,107,C15,PROD_D,5,15.0,2026-01-10,75.0,January,Saturday


In [131]:

df_sales['month'] = df_sales['transaction_date'].dt.month_name()
df_sales['day'] = df_sales['transaction_date'].dt.day_name()
merged = pd.merge(df_sales, df_customers, how='inner', on='customer_id')
merged['name'] = ['a', 'b', 'c', 'd']
merged

,order_id,customer_id,product_sku,quantity,unit_price,transaction_date,total,month,day,country,signup_date,name
0,101,C10,PROD_A,2,120.0,2026-01-05,240.0,January,Monday,USA,2025-05-12,a
1,102,C11,PROD_B,1,45.5,2026-01-06,45.5,January,Tuesday,USA,2025-08-19,b
2,103,C12,PROD_A,3,120.0,NaT,360.0,NaN,NaN,Canada,2025-11-01,c
3,105,C13,PROD_B,1,45.5,2026-01-08,45.5,January,Thursday,UK,2026-01-01,d


In [103]:
country_sum = merged.groupby('country')['total'].agg(
    total_sales='sum',
    avg='mean'
).sort_values(by= 'total_sales', ascending=False)

country_sum

,total_sales,avg
country,,
Canada,360.0,360.00
USA,285.5,142.75
UK,45.5,45.50


In [106]:
top_cus = merged.groupby('customer_id')['total'].agg(
    total='sum'
).sort_values(by= 'total',ascending=False)
top_cus

,total
customer_id,
C12,360.0
C10,240.0
C11,45.5
C13,45.5


In [125]:
target_id = 'C12'
customer = merged.loc[merged['customer_id'] == target_id, 'name'].values[0]
customer

'c'

In [132]:
merged = merged.resample('D', on= 'transaction_date')['total'].sum()


In [133]:
merged

transaction_date
2026-01-05    240.0
2026-01-06     45.5
2026-01-07      0.0
2026-01-08     45.5
Name: total, dtype: float64

🛒 E-Commerce Data Cleaning & Analytics Pipeline
📌 Project Overview
This project demonstrates an end-to-end Pandas data pipeline that ingests dirty, unstandardized sales and CRM data from multiple sources, performs data auditing and cleaning, applies feature engineering, and aggregates key business KPIs.

🛠️ Data Cleaning & Processing Steps
1. Data Ingestion & Auditing
Schema Audit: Inspected column data types using .info() and checked missing values using .isna().sum().

Identified Issues:

Irregular date formats (2026/01/06, Jan 07, 2026, string inputs).

Dirty numeric values formatted as currency strings ("$120.00").

Missing and invalid quantities (negative values indicating returns/errors).

Leading and trailing whitespace in categorical text fields (" UK ").

Duplicated order IDs (105).

2. Data Cleaning & Normalization
Currency Formatting: Stripped dollar signs ($) from unit_price and converted the column to float.

Numeric Sanitization: Handled bad values in quantity using pd.to_numeric(), converting invalid values to NaN. Filtered out invalid entries where quantity <= 0.

Date Parsing: Normalized mixed string formats in transaction_date into standardized datetime64 objects using pd.to_datetime(..., format='mixed').

String Trimming: Cleaned whitespace in categorical data using .str.strip() on the country column.

Deduplication: Removed redundant transaction entries using df.drop_duplicates(subset=['order_id']).

3. Feature Engineering & Merging
Calculated Column: Created total revenue per line item (quantity * unit_price).

Time Features: Extracted temporal dimensions (month and day) from normalized datetime objects using .dt.month_name() and .dt.day_name().

Relational Join: Combined transactions (df_sales) and user profiles (df_customers) using an inner join on customer_id.

4. KPI Aggregation & Reporting
Grouped the merged dataset by country using .groupby() and .agg().

Computed total revenue (sum) and Average Order Value (AOV) (mean) simultaneously.

Sorted the output by total revenue in descending order to identify key markets.

💡 Tech Stack
Language: Python

Library: Pandas, NumPy

Environment: Jupyter Notebook / Visual Studio Code